In [25]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.utils.data import random_split,DataLoader

In [10]:

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [11]:
os.makedirs("models",exist_ok=True)
os.makedirs("outputs",exist_ok=True)

In [12]:
BATCH_SIZE=64
EPOCHS=5
LEARNING_RATE=0.001

In [13]:
#Load temporarily for statistics

dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

loader = DataLoader(dataset, batch_size=60000)

data, _ = next(iter(loader))

mean = data.mean()
std = data.std()

print(mean, std)

tensor(0.1307) tensor(0.3081)


In [14]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3081,))
])

In [15]:
dataset=datasets.MNIST(
    root='./data',
    train=True,
    transform=transform,
    download=True
)

test_data=datasets.MNIST(
    root='./data',
    train=False,
    transform=transform,
    download=True
)

In [16]:
print(f"Len of train dataset {len(dataset)}")
print(f"Len of test dataset {len(test_data)}")


Len of train dataset 60000
Len of test dataset 10000


In [17]:
train_size=int(0.8*len(dataset))
val_size=len(dataset)-train_size

train_dataset,val_dataset=random_split(
    dataset,
    [train_size,val_size]
)

In [18]:
train_loader=DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader=DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader=DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [22]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN,self).__init__()

        self.conv1=nn.Conv2d(
            in_channels=1,
            out_channels=32,
            padding=1,
            kernel_size=3
        )

        self.conv2=nn.Conv2d(
            in_channels=32,
            out_channels=64,
            padding=1,
            kernel_size=3
        )

        self.relu=nn.ReLU()
        self.pool=nn.MaxPool2d(kernel_size=2)

        self.fc1=nn.Linear(64*7*7,128)
        self.fc2=nn.Linear(128,10)

        self.dropout=nn.Dropout(0.25)

    def forward(self,x):

        x=self.conv1()
        x=self.relu()
        x=self.pool()

        x=self.conv2()
        x=self.relu()
        x=self.pool()

        x=x.view(x.size(0),-1)

        x=x.fc1()

        x=x.relu()
        x=x.dropout()

        x=x.fc2()

        return x





In [23]:
model=SimpleCNN().to(device=device)

In [26]:
criterion=nn.CrossEntropyLoss()

optimizer=optim.Adam(
    model.parameters(),
    LEARNING_RATE,

)